# Swin2SR LoRA — direct vs ADMM (10 BPP-diverse images)

Runs:
- `runs/bitrate_sr_swin_direct_10` — tags `*_direct`
- `runs/bitrate_sr_swin_admm_10` — tags `*_admm`

List: `image_lists/bpp_diverse_10.txt`

In [ ]:
from pathlib import Path
import json
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path("/gpfs/gpfs0/timofey.glukhikh/Science_Phan")
if not PROJECT.exists():
    PROJECT = Path("/gpfs/data/gpfs0/timofey.glukhikh/Science_Phan")

ROOTS = {
    "direct": PROJECT / "runs" / "bitrate_sr_swin_direct_10",
    "admm": PROJECT / "runs" / "bitrate_sr_swin_admm_10",
}
BACKBONE = "swin"
LAMS = ["0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.8", "1"]


def run_dir(root: Path, img: str, lam: str, method: str) -> Path:
    return root / f"{img}_{BACKBONE}_psnr35_lam{lam}_r4_{method}"


def list_images(root: Path, method: str) -> list[str]:
    return sorted({
        p.name.split(f"_{BACKBONE}_")[0]
        for p in root.glob(f"img*_{BACKBONE}_psnr35_lam*_r4_{method}")
        if (p / "metrics.json").exists()
    })


def mean_bpp(root: Path, img: str, method: str) -> float:
    vals = []
    for lam in LAMS:
        mf = run_dir(root, img, lam, method) / "metrics.json"
        if mf.exists():
            vals.append(json.loads(mf.read_text())["Bpp"])
    return float(np.mean(vals)) if vals else float("inf")


def load_rows(root: Path, method: str):
    rows = []
    for img in list_images(root, method):
        for lam in LAMS:
            mf = run_dir(root, img, lam, method) / "metrics.json"
            if not mf.exists():
                continue
            m = json.loads(mf.read_text())
            m["_img"] = img
            m["_method"] = method
            rows.append(m)
    return rows


for method, root in ROOTS.items():
    print(f"{method:8s} {root}  exists={root.exists()}  n_dirs={len(list(root.glob('img*')) if root.exists() else [])}")

## Bpp training curves — direct

In [ ]:
def plot_bpp_curves(method: str):
    root = ROOTS[method]
    imgs = list_images(root, method)
    if not imgs:
        print(f"No runs for {method} under {root}")
        return
    imgs = sorted(imgs, key=lambda im: mean_bpp(root, im, method), reverse=True)
    n_r, n_c = len(imgs), len(LAMS)
    fig, axes = plt.subplots(n_r, n_c, figsize=(2.0 * n_c, 1.5 * n_r), sharex=True, sharey=False)
    if n_r == 1:
        axes = np.array([axes])
    for i, img in enumerate(imgs):
        for j, lam in enumerate(LAMS):
            ax = axes[i, j]
            cpath = run_dir(root, img, lam, method) / "curves.npz"
            if not cpath.exists():
                ax.set_axis_off()
                continue
            b = np.load(cpath)["bpp_values"]
            ax.plot(b, lw=0.8)
            ax.grid(True, alpha=0.3)
            if i == 0:
                ax.set_title(f"λ={lam}", fontsize=8)
            if j == 0:
                ax.set_ylabel(f"{img}\n({mean_bpp(root, img, method):.3f})", fontsize=7)
            if i == n_r - 1:
                ax.set_xlabel("step", fontsize=8)
    fig.suptitle(f"Swin {method}: Bpp curves (rows=images by mean Bpp)", y=1.01)
    fig.tight_layout()
    plt.show()


plot_bpp_curves("direct")

## Bpp training curves — ADMM

In [ ]:
plot_bpp_curves("admm")

## Mean RD: direct vs ADMM

In [ ]:
def mean_rd(rows):
    by_lam = defaultdict(list)
    for m in rows:
        lam, bpp, psnr = m.get("lambda"), m.get("Bpp"), m.get("PSNR_cmpref")
        if lam is None or bpp is None or psnr is None:
            continue
        by_lam[float(lam)].append((float(bpp), float(psnr)))
    lams = sorted(by_lam)
    xs, ys, ns = [], [], []
    for lam in lams:
        pts = by_lam[lam]
        xs.append(float(np.mean([p[0] for p in pts])))
        ys.append(float(np.mean([p[1] for p in pts])))
        ns.append(len(pts))
    return lams, xs, ys, ns


plt.figure(figsize=(7, 4.5))
styles = {"direct": ("o-", "C0"), "admm": ("s--", "C1")}
for method, (fmt, color) in styles.items():
    rows = load_rows(ROOTS[method], method)
    if not rows:
        print(f"skip {method}: no metrics")
        continue
    lams, xs, ys, ns = mean_rd(rows)
    print(f"\n=== Swin {method} (n per λ) ===")
    print(f"{'lam':>6s} {'Bpp':>8s} {'PSNRref':>8s} {'n':>4s}")
    for lam, x, y, n in zip(lams, xs, ys, ns):
        print(f"{lam:6.3g} {x:8.4f} {y:8.3f} {n:4d}")
    order = np.argsort(xs)
    xs_o = [xs[i] for i in order]
    ys_o = [ys[i] for i in order]
    lams_o = [lams[i] for i in order]
    plt.plot(xs_o, ys_o, fmt, color=color, markersize=7, label=method)
    for x, y, lam in zip(xs_o, ys_o, lams_o):
        plt.annotate(f"λ={lam:g}", (x, y), textcoords="offset points", xytext=(3, 3), fontsize=7, color=color)

plt.xlabel("Bpp (mean over images)")
plt.ylabel("PSNR vs GT compressed (mean)")
plt.title("Swin LoRA — direct vs ADMM (10 images)")
plt.grid(True, alpha=0.3)
plt.legend()
out = ROOTS["direct"].parent / "rd_swin_direct_vs_admm_10.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150, bbox_inches="tight")
print(f"Saved -> {out}")
plt.show()